## Azure ML Data Stores and Data Sets

### Set up the Azure ML Client

In [1]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

print(f"Ready to use Azure ML SDK v2 to work with {ml_client.workspace_name}")

Found the config file in: /config.json


Ready to use Azure ML SDK v2 to work with <WORKSPACE_NAME>


### View all Default Datastores in your workspace

In [2]:
# Get the default datastore
default_ds = ml_client.datastores.get_default()

# List all datastores and check which one is default
for ds in ml_client.datastores.list():
    print(ds.name, "- Default =", ds.name == default_ds.name)

azureml_globaldatasets - Default = False
aimldatastoreidk - Default = True
workspaceworkingdirectory - Default = False
workspaceartifactstore - Default = False
workspacefilestore - Default = False
workspaceblobstore - Default = False


### Connect a new Azure Blob Storage account as a Datastore

In [ ]:
aml_datastore = AzureBlobDatastore(
    name="aimldatastoreidk",
    description="Datastore pointing to a blob container using https protocol.",
    account_name="aimldatastoreidk",
    container_name="datastore",
    protocol="https",
    credentials=AccountKeyConfiguration(
        account_key=""
    ),
)

ml_client.create_or_update(aml_datastore)

### Getting the Default Datastore

In [4]:
# Get the default datastore
default_ds = ml_client.datastores.get_default()

# Enumerate all datastores and indicate which one is the default
for ds in ml_client.datastores.list():
    print(ds.name, "- Default =", ds.name == default_ds.name)

azureml_globaldatasets - Default = False
aimldatastoreidk - Default = True
workspaceworkingdirectory - Default = False
workspaceartifactstore - Default = False
workspacefilestore - Default = False
workspaceblobstore - Default = False


### Register Data Asset

In [ ]:
import requests
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Upload file to datastore path
from azure.ai.ml.operations import DatastoreOperations

datastore = ml_client.datastores.get_default()

# Construct the datastore path
datastore_path = f"azureml://datastores/{datastore.name}/paths/diabetes.csv"

# Register the uploaded file as a data asset
data_asset = Data(
    path=datastore_path,
    type=AssetTypes.URI_FILE,
    name="diabetes-data",
    description="Diabetes dataset uploaded to datastore"
)

ml_client.data.create_or_update(data_asset)

In [6]:
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Get datastore
datastore = ml_client.datastores.get_default()

#constructing File URI
file_uri = (
    f"azureml://subscriptions/{ml_client.subscription_id}"
    f"/resourcegroups/{ml_client.resource_group_name}"
    f"/workspaces/{ml_client.workspace_name}"
    f"/datastores/{datastore.name}"
    f"/paths/diabetes.csv"
)

# Path to CSV in datastore
paths = [{
    "file": file_uri
}]

# Create MLTable from CSV
tbl = mltable.from_delimited_files(
    paths=paths,
    delimiter=",",
    header=MLTableHeaders.all_files_same_headers,
    infer_column_types=True,
    encoding=MLTableFileEncoding.utf8
)

# Preview dataset
print(tbl.show())

# Save MLTable definition locally
mltable_folder = "./diabetes_mltable"
tbl.save(mltable_folder)

# Register MLTable as data asset
data_asset = Data(
    path=mltable_folder,
    type=AssetTypes.MLTABLE,
    name="diabetes-mltable",
    description="Diabetes dataset stored as MLTable"
)

ml_client.data.create_or_update(data_asset)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (2.7.1) and mlflow-skinny (2.22.5) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument 

    Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0             6      148             72             35        0  33.6   
1             1       85             66             29        0  26.6   
2             8      183             64              0        0  23.3   
3             1       89             66             23       94  28.1   
4             0      137             40             35      168  43.1   
5             5      116             74              0        0  25.6   
6             3       78             50             32       88  31.0   
7            10      115              0              0        0  35.3   
8             2      197             70             45      543  30.5   
9             8      125             96              0        0   0.0   
10            4      110             92              0        0  37.6   
11           10      168             74              0        0  38.0   
12           10      139             80            

Data({'path': 'azureml://subscriptions/<SUBSCRIPTION_ID>/resourcegroups/<RESOURCE_GROUP>/workspaces/<WORKSPACE_NAME>/datastores/workspaceblobstore/paths/LocalUpload/2f11c14fefc3159b5673ba436ea5a0af42b27665c1a7e03ed6bc63c33f235520/diabetes_mltable/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['azureml://subscriptions/<SUBSCRIPTION_ID>/resourcegroups/<RESOURCE_GROUP>/workspaces/<WORKSPACE_NAME>/datastores/aimldatastoreidk/paths/diabetes.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-mltable', 'description': 'Diabetes dataset stored as MLTable', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/<SUBSCRIPTION_ID>/resourceGroups/<RESOURCE_GROUP>/providers/Microsoft.MachineLearningServices/workspaces/<WORKSPACE_NAME>/data/diabetes-mltable/versions/2', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/<RESOURCE_GROU

In [7]:
print("Datasets:")

for data_asset in ml_client.data.list():
    print("\t", data_asset.name, "version", data_asset.version)

Datasets:
	 diabetes-data version None
	 diabetes-mltable version None
	 sales-data version None
	 MD-automobilepipeline-Train_Model-Trained_model-da777bdc version None
	 TD-automobilepipeline-Clean_Missing_Data-Cleaning_transformation-fc361501 version None
